<a href="https://colab.research.google.com/github/chriski777/mech_interp/blob/main/ji_an_2025_small_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Neurofeedback Task

This notebook tries to reproduce and build off of Ji-An et al. (2025), *Language Models Are Capable of Metacognitive Monitoring and Control of Their Internal Activations* ([arXiv:2505.13763](https://arxiv.org/abs/2505.13763)) on a smaller model that we can run off of Colab GPU.



## Setup

In [ ]:
%pip install -q "transformer_lens>=4.0,<5" scikit-learn

### Initialize Settings

Import relevant packages

In [ ]:
import importlib.metadata as md
from packaging.version import Version
import transformers, torch
import os
import numpy as np
import pandas as pd
import yaml

from google.colab import userdata

installed = Version(md.version("transformers"))
if Version(transformers.__version__) != installed:
    raise RuntimeError("An old copy of transformers is still loaded. "
                       "Runtime → Restart session, then run from this cell.")
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU."
print("transformers", installed, "| transformer_lens", md.version("transformer_lens"),
      "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Settings ---------------------------------------------------------------
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct" # Model name to fetch
LAYERS = [0, 3, 7, 11, 15]     # the paper's layers for this model (0/25/50/75/100% depth)
REPORT_LAYERS = [7]

DTYPE = torch.float16
AXES = ["LR", "PC1", "PC4", "PC32", "PC128", "PC512"]

RUN_SHUFFLED_CONTROL = True

N_EXAMPLES = 200               # labeled sentences per prompt (paper: 500)
N_REPEATS = 10                 # random sentence samples per condition (paper: 100)
SEED = 42
DATE_STRING = "26 Jul 2024"    #

USE_DRIVE = True               # save results to Google Drive so a disconnect doesn't lose them

Load in hugging face access token and initialize directory for saving results

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/neurofeedback_llama1b"
else:
    SAVE_DIR = "/content/neurofeedback_llama1b"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to", SAVE_DIR)

## Load dataset

We use the ETHICS commonsense-morality sentences and the prompt text saved in the authors' repository.


### Download dataset

In [ ]:
# Uncomment the code below to copy the sentences and prompts from the author


#!git clone -q --depth 1 https://github.com/sakimarquis/llm_neurofeedback.git /content/llm_neurofeedback 2>/dev/null || echo "Repo already cloned."
# import shutil
# from pathlib import Path

# SRC = Path(REPO)        # /content/llm_neurofeedback
# DST = Path(SAVE_DIR)

# FILES = [
#     "data/ethics_commonsense/cm_train.csv",
#     "data/ethics_commonsense/cm_test.csv",
#     "configs/prompts.yml",
#     "configs/meta_prompts.yml",
# ]

# for rel in FILES:
#     src, dst = SRC / rel, DST / rel
#     dst.parent.mkdir(parents=True, exist_ok=True)
#     shutil.copy2(src, dst)
#     print(f"Copied {rel} -> {dst}")

### Load sentences and prompts

Following the authors' `load_commonsense`, we use only the short scenarios (one or two sentences each) and interleave morally acceptable (label 0) and unacceptable (label 1) examples.

We select `n_sample = 1500` sentences. We use 900 samples to fit the axes and 600 samples for reporting.  Like the authors, we don't shuffle within labels, so the selection of train/test samples is deterministic.

In [ ]:
DATA_DIR = SAVE_DIR

def load_commonsense(data_dir, n_sample=1500, train_ratio=0.6):
    # Mirrors data/load.py::load_commonsense in the authors' repo.
    df = pd.concat([pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_train.csv"),
                    pd.read_csv(f"{data_dir}/data/ethics_commonsense/cm_test.csv")], ignore_index=True)
    df = df[df["is_short"] == True]
    zeros, ones = df[df["label"] == 0], df[df["label"] == 1]
    n = min(len(zeros), len(ones))
    rows = [r for pair in zip(zeros.iloc[:n].itertuples(), ones.iloc[:n].itertuples()) for r in pair]
    rows = rows[:n_sample]
    sentences = [r.input.strip() for r in rows]   # Llama's chat template trims whitespace anyway
    labels = np.array([r.label for r in rows])
    split = int(n_sample * train_ratio)
    return sentences[:split], labels[:split], sentences[split:], labels[split:]

train_sents, train_moral, test_sents, test_moral = load_commonsense(DATA_DIR)

# Just making sure that the moral scores are not in any of the the sentences
assert not any("[Score:" in s for s in train_sents + test_sents)

# Load all the prompts and meta prompts
with open(f"{DATA_DIR}/configs/prompts.yml") as f:
    prompts = yaml.safe_load(f)
with open(f"{DATA_DIR}/configs/meta_prompts.yml") as f:
    meta_prompts = yaml.safe_load(f)

SYSTEM_MSG, USER_MSG = prompts["system"], prompts["user_msg"]
REPORT_SYSTEM = (meta_prompts["report_NF"][0]["content"]
                 .replace("<SCORE_SET>", "0, 1").replace("<LOWEST>", "0").replace("<HIGHEST>", "1"))

print(f"{len(train_sents)} axis-fitting sentences, {len(test_sents)} reporting sentences")

example_idx = np.random.randint(len(train_sents))
print("\nExample sentence. :", train_sents[example_idx], 'Score:', train_moral[example_idx])
print("\nReporting-task system prompt:\n", REPORT_SYSTEM)

### Helper functions for tokenizing sentences

In [ ]:
from bisect import bisect_left, bisect_right
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def render(messages):
    # Remove date string templates
    return tokenizer.apply_chat_template(messages, tokenize=False, date_string=DATE_STRING)

def _n_tokens(text):
    # count number of tokens
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def locate(text, char_spans):
    # Returns in token psotopms
    # char_spans is the positions in the text string
    # (each index is a tuple of the character start and character end)
    # enc has token_ids, attention_mask (all 1's; don't matter), offset_mapping (start,end of text)  for each token
    # with add_special_tokens set to false, we don't want the BOS token added.
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = enc["input_ids"]
    # Get the start position of each token
    starts = [s for s, _ in enc["offset_mapping"]]

    def same(a, b):
        return " ".join(a.split()) == " ".join(b.split())

    def ok(spans):
        return all(same(tokenizer.decode(ids[s:e], clean_up_tokenization_spaces=False), text[cs:ce])
                   for (s, e), (cs, ce) in zip(spans, char_spans))
    spans = [(bisect_right(starts, cs) - 1, bisect_left(starts, ce)) for cs, ce in char_spans]
    if not ok(spans):
        spans = [(_n_tokens(text[:cs]), _n_tokens(text[:ce])) for cs, ce in char_spans]
        if not ok(spans):
            raise ValueError("Could not map text spans to tokens for this tokenizer.")
    return ids, spans

# Quick check on one sentence
_t = render([{"role": "system", "content": SYSTEM_MSG}, {"role": "user", "content": USER_MSG},
             {"role": "assistant", "content": test_sents[0]}])
# Retrieve character span indices
_cs = _t.rfind(test_sents[0])
# Convert into token indices
_ids, [(_s, _e)] = locate(_t, [(_cs, _cs + len(test_sents[0]))])
print(repr(_t))
print("Pooled tokens:", tokenizer.convert_ids_to_tokens(_ids[_s:_e])) # text stored as bytes

## Record activations using TransformerLens

Here we use transformerLens model' wrapper to load the downloaded model's weights.


In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(MODEL_NAME, device="cuda", dtype=DTYPE)
N_LAYERS, D_MODEL = bridge.cfg.n_layers, bridge.cfg.d_model # (number of transformer blocks, width of the residual stream)
assert max(LAYERS) < N_LAYERS
print(f"{MODEL_NAME}: {N_LAYERS} layers, residual stream width {D_MODEL}")